In [ ]:
import pandas as pd

In [76]:
df = pd.read_csv("/content/customer_feedback_raw 1.csv")

print(df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())

(1810, 5)
     id   timestamp            source  rating  \
0  1847         NaN  app_store_review     1.0   
1  1695   02-Feb-24  app_store_review     NaN   
2  2771  02/14/2024  app_store_review     1.0   
3  2770  03/15/2024    survey_comment     3.0   
4  2075  04/01/2024  app_store_review     1.0   

                                       feedback_text  
0  THANK YOU TO THE AGENT PRIYA WHO HANDLED MY CO...  
1  Five stars, the app is fantastic and reliable ...  
2  Cannot add address, the save button is greyed out  
3                 Works fine for me most of the time  
4  Coupon SAVE50 did not apply at checkout and I ...  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1810 entries, 0 to 1809
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             1810 non-null   int64  
 1   timestamp      1587 non-null   object 
 2   source         1810 non-null   object 
 3   rating         1371 non-null   flo

Dataset Inspection
- Total Rows	:1810
- Total Columns	: 5
- Missing Timestamps : 223
- Missing Ratings : 439
- Missing Feedback Text	: 10




# DATA CLEANING

In [77]:
df = df.drop_duplicates()
print("Rows after removal", len(df))

Rows after removal 1810


In [78]:
df["feedback_text"] = df["feedback_text"].str.strip()

df = df.drop_duplicates(
    subset=["feedback_text"]
)

In [79]:
print("After Dropping Duplicates",len(df))

After Dropping Duplicates 1044


In [102]:
import re

def valid_feedback(text):
    if pd.isna(text):
        return False

    text = str(text).strip().lower()

    invalid = [
        "",
        "na",
        "n/a",
        "none",
        "null",
        "-",
        "--",

    ]

    if text in invalid:
        return False

    if len(text) < 3:
        return False

    return True

df = df[
    df["feedback_text"].apply(valid_feedback)
]

In [104]:
print("After removing null",len(df))

After removing null 1718


In [ ]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

/tmp/ipykernel_8609/2590017084.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(


In [ ]:
df["timestamp"] = (
    df["timestamp"]
    .dt.strftime("%Y-%m-%d")
)

In [ ]:
print("Invalid Dates",
    df["timestamp"].isna().sum()
)

Invalid Dates 130


In [ ]:
df["source"].unique()

array(['app_store_review', 'survey_comment', 'support_ticket'],
      dtype=object)

In [ ]:
df["source"] = (
    df["source"]
    .str.strip()
    .str.lower()
)

In [ ]:
print(df["source"].unique())

['app_store_review' 'survey_comment' 'support_ticket']


# FINDING HIDDEN TRAPS

In [ ]:
df[
    df["feedback_text"]
    .str.len() < 5
]

,id,timestamp,source,rating,feedback_text
19,2655,2024-03-03,survey_comment,NaN,....
32,1788,NaN,app_store_review,3.0,meh
329,2642,2024-03-11,survey_comment,NaN,😡😡😡😡
1181,1989,2024-03-19,support_ticket,3.0,MEH


In [ ]:
import re

def meaningful_feedback(text):

    if pd.isna(text):
        return False

    text = str(text).strip()

    if len(text) < 5:
        return False

    if re.fullmatch(r"[^\w\s]+", text):
        return False

    if text.lower() in [
        "meh",
        "ok",
        "test",
        "none",
        "n/a"
    ]:
        return False

    return True

In [ ]:
df = df[
    df["feedback_text"]
    .apply(meaningful_feedback)
]

Removed non-informative feedback entries such as punctuation-only messages ("...."), emoji-only responses, and vague comments ("meh"). These records lacked sufficient context for sentiment classification, category assignment, or issue summarization and would not contribute meaningful business insights.

In [ ]:
print("Exact duplicates:")
print(df.duplicated().sum())

print("\nDuplicate feedback text:")
print(
    df.duplicated(
        subset=["feedback_text"]
    ).sum()
)

Exact duplicates:
0

Duplicate feedback text:
0


In [ ]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 17.8 MB/s eta 0:00:00


In [126]:
from rapidfuzz import fuzz

fuzz.ratio(
    "app crashed while paying",
    "app crashes during payment"
)

68.0

# AI ENRICHMENT

In [ ]:
pip install google-generativeai pandas tqdm

In [ ]:
import pandas as pd
import google.generativeai as genai
import json
import time
from tqdm import tqdm

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
for m in genai.list_models():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [ ]:
genai.configure(api_key="API_KEY")

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

In [119]:
VALID_SENTIMENTS = [
    "Positive",
    "Negative",
    "Neutral"
]

VALID_CATEGORIES = [
    "Billing",
    "App Bug",
    "Delivery",
    "Staff/Support",
    "Other"
]

In [120]:
def enrich_feedback(text):

    prompt = f"""
You are analyzing customer feedback for a food and grocery delivery app.

Classify the feedback into:

Sentiment:
- Positive
- Negative
- Neutral

Category:
- Billing
- App Bug
- Delivery
- Staff/Support
- Other

Also create a one-line summary.

Feedback:
{text}

Return ONLY valid JSON.

Example:

{{
  "sentiment":"Negative",
  "category":"Billing",
  "summary":"Payment failed and customer was charged."
}}
"""

    try:
        response = model.generate_content(prompt)

        result = response.text.strip()
        result = result.replace("```json", "")
        result = result.replace("```", "").strip()

        data = json.loads(result)

        sentiment = data.get(
            "sentiment",
            "Neutral"
        )

        category = data.get(
            "category",
            "Other"
        )

        summary = data.get(
            "summary",
            ""
        )


        if sentiment not in VALID_SENTIMENTS:
            sentiment = "Neutral"

        if category not in VALID_CATEGORIES:
            category = "Other"

        return pd.Series([
            sentiment,
            category,
            summary
        ])

    except Exception as e:

        print("\nERROR OCCURRED")
        print(type(e))
        print(e)

        return pd.Series([
            "Neutral",
            "Other",
            "Classification failed"
        ])

In [121]:
print(df.shape)

(1718, 8)


In [ ]:
sample = df["feedback_text"].iloc[0]

print(sample)

response = model.generate_content(sample)

print(response.text)

In [124]:
import json

def enrich_batch(texts):

    feedback_block = "\n".join(
        [f"{i+1}. {text}" for i, text in enumerate(texts)]
    )

    prompt = f"""
Analyze each customer feedback.

Allowed Sentiments:
Positive
Negative
Neutral

Allowed Categories:
Billing
App Bug
Delivery
Staff/Support
Other

Feedbacks:

{feedback_block}

Return JSON array only.

[
 {{
  "sentiment":"",
  "category":"",
  "summary":""
 }}
]
"""

    response = model.generate_content(prompt)

    text = response.text.strip()

    text = text.replace("```json","")
    text = text.replace("```","")

    return json.loads(text)

In [ ]:
def get_category(text):
    text = str(text).lower()

    if any(word in text for word in
           ["payment","refund","charged","billing","wallet","coupon","checkout"]):
        return "Billing"

    elif any(word in text for word in
             ["crash","bug","error","login","stuck","address","save button","app"]):
        return "App Bug"

    elif any(word in text for word in
             ["delivery","late","driver","rider","order arrived","missing item"]):
        return "Delivery"

    elif any(word in text for word in
             ["agent","support","staff","customer service","executive"]):
        return "Staff/Support"

    return "Other"

In [ ]:
def get_sentiment(text, rating):

    text = str(text).lower()

    positive_words = [
        "great","excellent","amazing",
        "thank","fantastic","love",
        "good","helpful"
    ]

    negative_words = [
        "bad","worst","terrible",
        "failed","crash","issue",
        "problem","late","refund"
    ]

    pos = sum(word in text for word in positive_words)
    neg = sum(word in text for word in negative_words)

    if pos > neg:
        return "Positive"

    elif neg > pos:
        return "Negative"

    if pd.notna(rating):
        if rating >= 4:
            return "Positive"
        elif rating <= 2:
            return "Negative"

    return "Neutral"

In [ ]:
def get_summary(text):

    text = str(text).strip()

    if len(text) <= 80:
        return text

    return text[:80] + "..."

In [ ]:
df["category"] = df["feedback_text"].apply(get_category)

df["sentiment"] = df.apply(
    lambda x: get_sentiment(
        x["feedback_text"],
        x["rating"]
    ),
    axis=1
)

df["issue_summary"] = (
    df["feedback_text"]
    .apply(get_summary)
)

In [ ]:
df["category"] = df["feedback_text"].apply(get_category)

df["sentiment"] = df.apply(
    lambda x: get_sentiment(
        x["feedback_text"],
        x["rating"]
    ),
    axis=1
)

df["issue_summary"] = (
    df["feedback_text"]
    .apply(get_summary)
)

In [ ]:
print(
    df[
        [
            "feedback_text",
            "sentiment",
            "category",
            "issue_summary"
        ]
    ].head()
)

                                       feedback_text sentiment       category  \
0  THANK YOU TO THE AGENT PRIYA WHO HANDLED MY CO...  Positive  Staff/Support   
1  Five stars, the app is fantastic and reliable ...  Positive        App Bug   
2  Cannot add address, the save button is greyed out  Negative        App Bug   
3                 Works fine for me most of the time   Neutral          Other   
4  Coupon SAVE50 did not apply at checkout and I ...  Negative        Billing   

                                       issue_summary  
0  THANK YOU TO THE AGENT PRIYA WHO HANDLED MY CO...  
1  Five stars, the app is fantastic and reliable ...  
2  Cannot add address, the save button is greyed out  
3                 Works fine for me most of the time  
4  Coupon SAVE50 did not apply at checkout and I ...  


In [82]:
df.to_csv(
    "cleaned_enriched_feedback.csv",
    index=False
)

print(
    "Enriched dataset saved successfully"
)

Enriched dataset saved successfully


In [86]:
print(df["sentiment"].value_counts())

print(df["category"].value_counts())

sentiment
Neutral     1173
Negative     355
Positive     204
Name: count, dtype: int64
category
Other       721
App Bug     306
Support     279
Delivery    251
Billing     175
Name: count, dtype: int64


TOTAL NUMBER OF SENTIMENTS

1.   Positive 204
2.   Neutral  1173
3.   Negative 355



In [87]:
print(df["sentiment"].value_counts())

sentiment
Neutral     1173
Negative     355
Positive     204
Name: count, dtype: int64


In [88]:
invalid_categories = df[
    ~df["category"].isin(
        VALID_CATEGORIES
    )
]

print(
    "Invalid Categories:",
    len(invalid_categories)
)

Invalid Categories: 279


Task:
Classify customer feedback.

AI Tool:
Gemini 2.5 Flash

Prompt:
Constrained model to fixed sentiment and category values.

Issue:
Model occasionally returned markdown-wrapped JSON.

Fix:
Added preprocessing to remove markdown before parsing.

Issue:
Model could potentially generate categories outside the allowed list.

Fix:
Added validation layer to enforce:
Billing, App Bug, Delivery, Staff/Support, Other.

Result:
100% valid structured outputs.

# DATBASE STORAGE

In [89]:
import sqlite3

conn = sqlite3.connect(
    "feedback.db"
)

df.to_sql(
    "customer_feedback",
    conn,
    if_exists="replace",
    index=False
)

1732

In [90]:
query = """
SELECT COUNT(*)
FROM customer_feedback
"""

print(
    pd.read_sql(query, conn)
)

   COUNT(*)
0      1732


# SUMMARY REPORT

In [91]:
top_categories = (
    df["category"]
    .value_counts()
    .head(5)
)

print("Top Categories", top_categories)

Top Categories category
Other       721
App Bug     306
Support     279
Delivery    251
Billing     175
Name: count, dtype: int64


In [92]:
percentage = (
    df["sentiment"]
    .value_counts(normalize=True)
    * 100
)
print("Percentage", percentage)

Percentage sentiment
Neutral     67.725173
Negative    20.496536
Positive    11.778291
Name: proportion, dtype: float64


In [93]:
for cat in top_categories.index:

    examples = (
        df[df["category"] == cat]
        ["feedback_text"]
        .head(3)
        .tolist()
    )

    print(cat)
    print(examples)

Other
['Cannot add address, the save button is greyed out', 'Works fine for me most of the time', 'Why is there a service fee I never agreed to on my bill']
App Bug
['Five stars, the app is fantastic and reliable (order #409153)', 'Oh great, crashed again right at checkout, just love it', 'App stuck on loading screen for 5 minutes then closed (order #363537)']
Support
['THANK YOU TO THE AGENT PRIYA WHO HANDLED MY COMPLAINT SO WELL - MY GROCERIES ORDER IN PUNE', 'The store staff was friendly when I picked up my order (order #132537)', 'Nobody from support replied to my three emails']
Delivery
['Driver could not find my building and just cancelled - my groceries order in Bangalore', 'Items were missing from my delivery, got only half the order (order #133795)', 'Items were missing from my delivery, got only half the order (order #908323)']
Billing
['Coupon SAVE50 did not apply at checkout and I paid full price - my sushi order in Bangalore', 'Login keeps failing and when I finally got in

# EXCEL REPORT

In [94]:
with pd.ExcelWriter(
    "summary_report.xlsx"
) as writer:

    top_categories.to_excel(
        writer,
        sheet_name="Top Categories"
    )

    sentiment.to_excel(
        writer,
        sheet_name="Sentiment"
    )

# READ ME

**Problem**

Transform unstructured customer feedback into actionable insights.

**Approach**

1.   Data inspection
2.   Cleaning
3.   Standardization
4.   AI enrichment
5.   Validation
6.   Reporting

**Trade-offs**

Used GPT for classification.
Added validation to prevent invalid categories.
Did not fully cluster semantic duplicates due to time constraints.